In [4]:
import pandas as pd
from pathlib import Path

# Find the data folder automatically
data_dir = Path("data") if Path("data").exists() else Path("../data")
print("Reading from:", data_dir.resolve())

orders = pd.read_csv(data_dir / "olist_orders_dataset.csv")
customers = pd.read_csv(data_dir / "olist_customers_dataset.csv")
items = pd.read_csv(data_dir / "olist_order_items_dataset.csv")
payments = pd.read_csv(data_dir / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(data_dir / "olist_order_reviews_dataset.csv")
products = pd.read_csv(data_dir / "olist_products_dataset.csv")

tables = {"orders": orders, "customers": customers, "items": items,
          "payments": payments, "reviews": reviews, "products": products}

for name, df in tables.items():
    print(f"\n--- {name} ---")
    print("Rows, columns:", df.shape)
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print("Missing values:\n", missing if not missing.empty else "None")
    print(df.head(2))

Reading from: /Users/vijay/Documents/olist-ecommerce-sales-analysis/data

--- orders ---
Rows, columns: (99441, 8)
Missing values:
 order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08-07 15:27:45   

  order_estimated_delivery_date  
0           2017-10-18 00:00:00  
1           2018-08-13 00:00:00  

--- customers ---
Rows, columns:

In [5]:
for name, df in tables.items():
    print(f"{name:10} rows: {df.shape[0]:>7,}  columns: {df.shape[1]}")

orders     rows:  99,441  columns: 8
customers  rows:  99,441  columns: 5
items      rows: 112,650  columns: 7
payments   rows: 103,886  columns: 5
reviews    rows:  99,224  columns: 7
products   rows:  32,951  columns: 9


In [6]:
# 1. Convert date columns from text to real dates
date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# 2. Check order statuses
print(orders["order_status"].value_counts(), "\n")

# 3. Keep delivered orders only
delivered = orders[orders["order_status"] == "delivered"].copy()
delivered = delivered.dropna(subset=["order_delivered_customer_date"])

# 4. Create new columns
delivered["order_month"] = delivered["order_purchase_timestamp"].dt.to_period("M")
delivered["delivery_days"] = (delivered["order_delivered_customer_date"]
                              - delivered["order_purchase_timestamp"]).dt.days
delivered["is_late"] = (delivered["order_delivered_customer_date"]
                        > delivered["order_estimated_delivery_date"])

# 5. Check duplicates
print("Duplicate orders:", orders["order_id"].duplicated().sum())
print("Delivered orders kept:", len(delivered))
print("Late deliveries:", f"{delivered['is_late'].mean():.1%}")
print("Average delivery days:", round(delivered["delivery_days"].mean(), 1))

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64 

Duplicate orders: 0
Delivered orders kept: 96470
Late deliveries: 8.1%
Average delivery days: 12.1
